# Step 1: GEE接続確認 + 1圃場のNDVI取得

**目的:** GEEに接続できるかを確認し、1つの圃場（field_id=1）のNDVI時系列を取得します。

**処理の流れ:**
1. GEE認証・初期化
2. 1圃場の座標を指定
3. Sentinel-2 + Cloud Score+ でNDVI時系列を取得
4. 結果をDataFrameで表示・簡単なプロット


## Cell 1: 認証

**初回のみ** `ee.Authenticate()` を実行します。
ブラウザが開くのでGEEに登録したGoogleアカウントでログインしてください。
2回目以降は不要です。


In [ ]:
import ee

# 初回のみ実行（ブラウザが開きます）
ee.Authenticate()


In [ ]:
# GCP プロジェクト ID を設定して初期化
# 自分のGCPプロジェクトIDに書き換えてください
GCP_PROJECT = 'YOUR_GCP_PROJECT_ID'  # <- ここを変更

ee.Initialize(project=GCP_PROJECT)
print('GEE接続成功！')
print(f'  使用プロジェクト: {GCP_PROJECT}')


## Cell 2: 圃場座標の読み込み


In [ ]:
import sqlite3
import pandas as pd

FIELD_DB = '../../data/processed/FieldData_fieldid.db'

conn = sqlite3.connect(FIELD_DB)
fields_df = pd.read_sql("""
    SELECT field_id, year, lat, lon, yield
    FROM Questionaire
    WHERE lat IS NOT NULL AND lon IS NOT NULL AND yield IS NOT NULL
    ORDER BY field_id, year
""", conn)
conn.close()

fields_df['field_id'] = fields_df['field_id'].astype(int)
fields_df['year']     = fields_df['year'].astype(int)
fields_df['lat']      = pd.to_numeric(fields_df['lat'], errors='coerce')
fields_df['lon']      = pd.to_numeric(fields_df['lon'], errors='coerce')

print(f'圃場数: {fields_df["field_id"].nunique()} 圃場')
print(f'対象年: {sorted(fields_df["year"].unique().tolist())}')
fields_df.head()


## Cell 3: NDVI取得関数

**使用するGEEデータセット:**
- `COPERNICUS/S2_SR_HARMONIZED`: Sentinel-2 地表反射率
- `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED`: AIベース雲評価

**NDVI = (B8 - B4) / (B8 + B4)**


In [ ]:
def get_ndvi_timeseries(lat, lon, year, buffer_m=30, cs_threshold=0.65):
    """
    指定した点座標周辺のSentinel-2 NDVI時系列を取得する。

    Args:
        lat (float)        : 緯度
        lon (float)        : 経度
        year (int)         : 対象年
        buffer_m (int)     : バッファ半径[m]（デフォルト30m）
        cs_threshold(float): Cloud Score+ 閾値（0〜1）

    Returns:
        pd.DataFrame: date, NDVI, n_pixels 列
    """
    # 点 -> 円形バッファに拡張
    point = ee.Geometry.Point([lon, lat])
    aoi   = point.buffer(buffer_m)

    # 大豆生育期間
    start_date = f'{year}-06-01'
    end_date   = f'{year}-11-30'

    # Cloud Score+ コレクション
    cs_col = (
        ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
        .filterDate(start_date, end_date)
        .filterBounds(aoi)
    )

    # Sentinel-2 SR コレクション（Cloud Score+ でマスク）
    s2_col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(start_date, end_date)
        .filterBounds(aoi)
        .linkCollection(cs_col, ['cs_cdf'])
        .map(lambda img: img.updateMask(img.select('cs_cdf').gte(cs_threshold)))
    )

    # NDVI バンド追加
    def add_ndvi(img):
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        return img.addBands(ndvi)

    ndvi_col = s2_col.map(add_ndvi).select('NDVI')

    # 各画像から対象領域のNDVI平均・ピクセル数を抽出
    def extract_value(img):
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.count(), sharedInputs=True
            ),
            geometry=aoi,
            scale=10,
            bestEffort=True,
        )
        return ee.Feature(None, {
            'date':     ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'NDVI':     stats.get('NDVI_mean'),
            'n_pixels': stats.get('NDVI_count'),
        })

    fc = ndvi_col.map(extract_value).getInfo()

    rows = []
    for feat in fc['features']:
        p = feat['properties']
        rows.append({
            'date':     p.get('date'),
            'NDVI':     p.get('NDVI'),
            'n_pixels': p.get('n_pixels'),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df['date']     = pd.to_datetime(df['date'])
    df['NDVI']     = pd.to_numeric(df['NDVI'], errors='coerce')
    df['n_pixels'] = pd.to_numeric(df['n_pixels'], errors='coerce')
    return df.dropna(subset=['NDVI']).sort_values('date').reset_index(drop=True)


## Cell 4: 実行 — field_id=1, year=2017

まず1圃場だけ試します。


In [ ]:
TARGET_FIELD_ID = 1
TARGET_YEAR     = 2017

row = fields_df[
    (fields_df['field_id'] == TARGET_FIELD_ID) &
    (fields_df['year']     == TARGET_YEAR)
]

if row.empty:
    print(f'field_id={TARGET_FIELD_ID}, year={TARGET_YEAR} のデータが見つかりません')
else:
    lat = float(row.iloc[0]['lat'])
    lon = float(row.iloc[0]['lon'])
    print(f'対象圃場: field_id={TARGET_FIELD_ID}  year={TARGET_YEAR}')
    print(f'  緯度: {lat:.6f}  経度: {lon:.6f}')
    print(f'  実測収量: {float(row.iloc[0]["yield"]):.1f} kg/10a')
    print()
    print('GEEからNDVI取得中... (数秒~30秒程度かかります)')

    ndvi_df = get_ndvi_timeseries(lat, lon, TARGET_YEAR, buffer_m=30)

    print(f'\n取得完了！  {len(ndvi_df)} 件のデータポイント')
    print(ndvi_df.to_string())


## Cell 5: プロット

NDVI時系列をプロットして、大豆の生育曲線が確認できるか見てみます。
健全な大豆圃場なら7~8月にかけてNDVIが上昇し、9月以降に低下する単峰性の曲線になるはずです。


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

os.makedirs('../../outputs', exist_ok=True)

if 'ndvi_df' in dir() and not ndvi_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))

    # NDVI時系列
    ax1 = axes[0]
    ax1.scatter(ndvi_df['date'], ndvi_df['NDVI'],
                color='#2ecc71', s=60, zorder=3, label='NDVI',
                edgecolors='#27ae60', linewidths=0.8)
    ax1.plot(ndvi_df['date'], ndvi_df['NDVI'],
             color='#27ae60', alpha=0.5, lw=1.5, linestyle='--')
    ax1.set_title(f'NDVI時系列  [field_id={TARGET_FIELD_ID}, year={TARGET_YEAR}]',
                  fontsize=13, fontweight='bold')
    ax1.set_ylabel('NDVI')
    ax1.set_ylim(-0.1, 1.0)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax1.xaxis.set_major_locator(mdates.MonthLocator())
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # 有効ピクセル数（雲マスクの品質チェック）
    ax2 = axes[1]
    ax2.bar(ndvi_df['date'], ndvi_df['n_pixels'],
            width=5, color='#3498db', alpha=0.7, label='有効ピクセル数')
    ax2.set_ylabel('有効ピクセル数')
    ax2.set_xlabel('日付')
    ax2.set_title('Cloud Score+ マスク後の有効ピクセル数')
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
    ax2.xaxis.set_major_locator(mdates.MonthLocator())
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.suptitle('Sentinel-2 NDVI | Cloud Score+ 雲マスク適用済み',
                 fontsize=11, y=1.01, color='gray')
    plt.tight_layout()

    save_path = f'../../outputs/ndvi_test_field{TARGET_FIELD_ID}_{TARGET_YEAR}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'プロット保存: {save_path}')

    print('\n--- NDVI 基本統計 ---')
    print(f'  観測日数  : {len(ndvi_df)}')
    print(f'  NDVI 最大 : {ndvi_df["NDVI"].max():.3f}')
    print(f'  NDVI 平均 : {ndvi_df["NDVI"].mean():.3f}')
    print(f'  NDVI 最小 : {ndvi_df["NDVI"].min():.3f}')
else:
    print('ndvi_df が空です。Cell 4 を先に実行してください。')


## 次のステップ

このNotebookで確認すること:
- `GEE接続成功！` が表示されるか
- NDVI時系列が取得できるか（観測日数、値の範囲）
- プロットで大豆の生育曲線（夏にピーク）が見えるか

確認できたら次は `02_ndvi_all_fields.ipynb` で全圃場に拡張します。
